In [1]:
import os, re, json, pickle
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import BertTokenizerFast, BertModel

DATA = Path("/kaggle/input/piash-dpfn-mabsa-data")
SENTIC_XLSX  = Path("/kaggle/input/piash-senticnet/senticnet.xlsx")
CONCEPT_JSON = Path("/kaggle/input/piash-conceptnet-cache/conceptnet_cache/conceptnet_cache_all_splits_cleaned.json")

OUT = Path("/kaggle/working/ke_gcn_graph_pkls_final")
OUT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("OUT:", OUT)


2025-12-19 14:32:58.285617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766154778.466679      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766154778.521127      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

device: cuda
OUT: /kaggle/working/dpfn_graph_pkls_final


In [2]:
# ---- SenticNet ----
sentic_df = pd.read_excel(SENTIC_XLSX)
SENTIC = {}
for _, r in sentic_df.iterrows():
    c = r.get("CONCEPT", None)
    p = r.get("POLARITY INTENSITY", None)
    if isinstance(c, str):
        try:
            SENTIC[c.lower()] = float(p)
        except:
            pass
print("Sentic concepts:", len(SENTIC))

# ---- ConceptNet cache ----
with open(CONCEPT_JSON, "r", encoding="utf-8") as f:
    CONCEPT_CACHE = json.load(f)
print("Concept cache keys:", len(CONCEPT_CACHE))


Sentic concepts: 295292
Concept cache keys: 3983


In [3]:
NEG = {"not","no","never","n't","cannot","cant","don't","dont","won't","wont"}

def ke_gcn_words(tweet_with_T: str, aspect: str):
    left, right = [s.strip() for s in tweet_with_T.split("$T$")]
    return (left + " " + aspect + " " + right).split()

def norm_tok(w: str) -> str:
    w = w.lower()
    w = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", w)  # strip punctuation at ends
    return w

def aspect_indices(words, aspect):
    asp_tokens = [norm_tok(x) for x in aspect.split()]
    asp_set = set([x for x in asp_tokens if x])
    wl = [norm_tok(x) for x in words]
    idxs = [i for i,t in enumerate(wl) if t in asp_set]
    return idxs

def sentic_score(tok: str) -> float:
    t = norm_tok(tok)
    if not t:
        return 0.0
    return float(SENTIC.get(t, 0.0))


In [4]:
def read_tsv(path: Path, split: str):
    df = pd.read_csv(path, sep="\t")
    if split in ("train","dev"):
        df = df.rename({
            "#1 Label": "sentiment",
            "#2 ImageID": "image_id",
            "#3 String": "tweet_content",
            "#3 String.1": "target",
        }, axis=1).drop(["index"], axis=1)
    else:
        df = df.rename({
            "index": "sentiment",
            "#1 ImageID": "image_id",
            "#2 String": "tweet_content",
            "#2 String.1": "target",
        }, axis=1)
    return df

def load_ori_adj(split: str):
    p = DATA / f"{split}_ori_adj.pkl"
    with open(p, "rb") as f:
        return pickle.load(f)

train_df = read_tsv(DATA/"train.tsv", "train")
dev_df   = read_tsv(DATA/"dev.tsv",   "dev")
test_df  = read_tsv(DATA/"test.tsv",  "test")

train_ori = load_ori_adj("train")
dev_ori   = load_ori_adj("dev")
test_ori  = load_ori_adj("test")

print("Rows:", len(train_df), len(dev_df), len(test_df))
print("Adj :", len(train_ori), len(dev_ori), len(test_ori))

# sanity check alignment
i = 0
words = ke_gcn_words(train_df.iloc[i]["tweet_content"], train_df.iloc[i]["target"])
A = np.array(train_ori[i])
print("sample0 words:", len(words), "adj shape:", A.shape)
assert A.shape == (len(words), len(words))
print("✅ alignment ok")


Rows: 3179 1122 1037
Adj : 3179 1122 1037
sample0 words: 10 adj shape: (10, 10)
✅ alignment ok


In [5]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

concept_emb_cache = {}

@torch.no_grad()
def embed_concept_phrase(concept: str) -> torch.Tensor:
    """Mean-pooled BERT embedding of the concept phrase (context-free)."""
    concept = str(concept).strip().lower()
    if concept in concept_emb_cache:
        return concept_emb_cache[concept]
    if not concept:
        v = torch.zeros(768, device=device)
    else:
        enc = tokenizer(concept, return_tensors="pt", truncation=True, max_length=32).to(device)
        h = bert(**enc).last_hidden_state[0]   # [L,768]
        v = h.mean(dim=0)
    concept_emb_cache[concept] = v
    return v


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [6]:
@torch.no_grad()
def embed_word_nodes_and_aspect(sentence_with_aspect: str, aspect: str, ke_gcn_word_list):
    """
    Single BERT pass on sentence_with_aspect.
    - Builds word-node embeddings aligned to ke_gcn_word_list (whitespace split)
      using offset overlap (robust to BERT subwords).
    - Extracts aspect embedding from the same encoding using char-span -> token offsets.

    Returns:
      node_feats: [N,768] aligned to ke_gcn_word_list
      asp_vec:    [768] context-aware aspect embedding
    """
    # 1) BERT encode with offsets
    enc = tokenizer(
        sentence_with_aspect,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        return_offsets_mapping=True,
        add_special_tokens=True
    ).to(device)

    offsets = enc["offset_mapping"][0].tolist()  # list of (start,end)
    enc_wo = {k: v for k, v in enc.items() if k != "offset_mapping"}

    out = bert(**enc_wo).last_hidden_state[0]    # [L,768]

    # 2) Aspect span in chars (first occurrence)
    sent_low = sentence_with_aspect.lower()
    asp_low  = aspect.lower().strip()
    start = sent_low.find(asp_low)
    if start == -1:
        # fallback: CLS
        asp_vec = out[0]
    else:
        end = start + len(asp_low)
        asp_token_ids = [
            ti for ti, (s, e) in enumerate(offsets)
            if s >= start and e <= end and not (s == 0 and e == 0)  # skip specials/pads offsets (0,0)
        ]
        if len(asp_token_ids) == 0:
            asp_vec = out[0]
        else:
            asp_vec = out[asp_token_ids].mean(dim=0)

    # 3) Word node feats aligned to KE-GCN whitespace words by char spans
    # Build char spans of ke_gcn_word_list inside sentence_with_aspect
    # We'll scan left-to-right to find each word occurrence in order.
    s = sentence_with_aspect
    spans = []
    pos = 0
    for w in ke_gcn_word_list:
        w_str = str(w)
        # find next occurrence at/after pos
        idx = s.find(w_str, pos)
        if idx == -1:
            # fallback: try case-insensitive find
            idx = s.lower().find(w_str.lower(), pos)
        if idx == -1:
            spans.append(None)
            continue
        span = (idx, idx + len(w_str))
        spans.append(span)
        pos = idx + len(w_str)

    N = len(ke_gcn_word_list)
    node_feats = torch.zeros((N, out.size(-1)), device=device)
    counts = torch.zeros((N,), device=device)

    for wi, sp in enumerate(spans):
        if sp is None:
            continue
        ws, we = sp
        tok_ids = [
            ti for ti, (s0, e0) in enumerate(offsets)
            if s0 >= ws and e0 <= we and not (s0 == 0 and e0 == 0)
        ]
        if len(tok_ids) == 0:
            continue
        node_feats[wi] = out[tok_ids].mean(dim=0)
        counts[wi] = 1.0

    # if any word failed mapping, fallback to CLS for that word (rare but safe)
    for wi in range(N):
        if counts[wi].item() == 0.0:
            node_feats[wi] = out[0]

    return node_feats, asp_vec


In [7]:
def build_aff_adj_myway(words, aspect, ori_adj, node_feats):
    """
    Only assigns weights on dependency edges from ori_adj.
    Weight = 1 + s_i + s_j + cosine(node_i, node_j) * T
    where T=1 iff edge touches an aspect node.
    """
    A_syn = np.array(ori_adj, dtype=np.float32)
    N = A_syn.shape[0]
    assert N == len(words)
    assert node_feats.shape[0] == N

    wl = [norm_tok(w) for w in words]
    s = np.array([sentic_score(w) for w in words], dtype=np.float32)

    # negation flip along dependency edges
    src, dst = np.nonzero(A_syn)
    for i, j in zip(src.tolist(), dst.tolist()):
        if wl[i] in NEG and s[j] != 0:
            s[j] = -s[j]
        if wl[j] in NEG and s[i] != 0:
            s[i] = -s[i]

    asp_idx = set(aspect_indices(words, aspect))
    if len(asp_idx) == 0:
        asp_idx = {0}

    # all-pairs cosine matrix via normalize + matmul (fast, correct)
    x = F.normalize(node_feats, dim=-1)                    # [N,768]
    sim = torch.matmul(x, x.transpose(0,1)).clamp(-1, 1)   # [N,N]
    sim_np = sim.detach().cpu().numpy().astype(np.float32)

    A_aff = np.zeros((N, N), dtype=np.float32)
    for i, j in zip(src.tolist(), dst.tolist()):
        T = 1.0 if (i in asp_idx or j in asp_idx) else 0.0
        A_aff[i, j] = 1.0 + s[i] + s[j] + sim_np[i, j] * T

    np.fill_diagonal(A_aff, 1.0)
    return A_aff


In [8]:
@torch.no_grad()
def top5_concepts_by_cosine_context(asp_vec, aspect: str, k=5, max_candidates=30):
    cands = CONCEPT_CACHE.get(aspect.lower(), [])
    cands = cands[:max_candidates]
    if not cands:
        return [""] * k

    if torch.norm(asp_vec) < 1e-8:
        return [""] * k

    asp_n = F.normalize(asp_vec, dim=0)

    scored = []
    for c in cands:
        c = str(c).strip().lower()
        if not c:
            continue
        c_vec = embed_concept_phrase(c)
        if torch.norm(c_vec) < 1e-8:
            continue
        c_n = F.normalize(c_vec, dim=0)
        sim = F.cosine_similarity(asp_n, c_n, dim=0).item()
        scored.append((c, sim))

    scored.sort(key=lambda x: x[1], reverse=True)
    top = [c for c,_ in scored[:k]]
    while len(top) < k:
        top.append("")
    return top


In [9]:
def build_concept_graph_adj(words, aspect, ori_adj, top_concepts, k=5):
    """
    A_con_graph: (N+5)x(N+5)
      - top-left NxN = ori syntactic adjacency
      - connect each concept node to aspect word indices
      - self loops
    """
    A_syn = np.array(ori_adj, dtype=np.float32)
    N = A_syn.shape[0]
    assert N == len(words)

    asp_idxs = aspect_indices(words, aspect)
    if len(asp_idxs) == 0:
        asp_idxs = [0]

    A = np.zeros((N + k, N + k), dtype=np.float32)
    A[:N, :N] = A_syn

    for ci in range(k):
        c = top_concepts[ci] if ci < len(top_concepts) else ""
        if not c:
            continue
        node_c = N + ci
        for ai in asp_idxs:
            A[ai, node_c] = 1.0
            A[node_c, ai] = 1.0

    np.fill_diagonal(A, 1.0)
    return A


In [10]:
def build_and_save_split(split_name, df, ori_list, k=5, max_candidates=30):
    aff_list = []
    con_graph_list = []
    top5_list = []

    for i in tqdm(range(len(df)), desc=f"Build PKLs: {split_name}"):
        tweetT = df.iloc[i]["tweet_content"]
        aspect = df.iloc[i]["target"]

        # KE-GCN word nodes (must match ori_adj)
        words = ke_gcn_words(tweetT, aspect)
        N = len(words)

        A_syn = np.array(ori_list[i], dtype=np.float32)
        assert A_syn.shape == (N, N)

        # sentence with aspect inserted
        sent_with_asp = tweetT.replace("$T$", aspect)

        # ONE encoding => word node feats + context-aware aspect embedding
        node_feats, asp_vec = embed_word_nodes_and_aspect(sent_with_asp, aspect, words)

        # affective adjacency (your method)
        A_aff = build_aff_adj_myway(words, aspect, A_syn, node_feats)

        # choose top-5 concepts by cosine to asp_vec
        top5 = top5_concepts_by_cosine_context(asp_vec, aspect, k=k, max_candidates=max_candidates)

        # concept graph adjacency (N+5 x N+5)
        A_con_graph = build_concept_graph_adj(words, aspect, A_syn, top5, k=k)

        aff_list.append(A_aff)
        con_graph_list.append(A_con_graph)
        top5_list.append(top5)

    aff_path = OUT / f"{split_name}_aff_adj.pkl"
    con_path = OUT / f"{split_name}_con_graph_adj_k{k}.pkl"
    top_path = OUT / f"{split_name}_top5_concepts_k{k}.pkl"

    with open(aff_path, "wb") as f:
        pickle.dump(aff_list, f)
    with open(con_path, "wb") as f:
        pickle.dump(con_graph_list, f)
    with open(top_path, "wb") as f:
        pickle.dump(top5_list, f)

    print("Saved:", aff_path)
    print("Saved:", con_path)
    print("Saved:", top_path)

build_and_save_split("train", train_df, train_ori, k=5, max_candidates=30)
build_and_save_split("dev",   dev_df,   dev_ori,   k=5, max_candidates=30)
build_and_save_split("test",  test_df,  test_ori,  k=5, max_candidates=30)


Build PKLs: train:   0%|          | 0/3179 [00:00<?, ?it/s]

Saved: /kaggle/working/dpfn_graph_pkls_final/train_aff_adj.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/train_con_graph_adj_k5.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/train_top5_concepts_k5.pkl


Build PKLs: dev:   0%|          | 0/1122 [00:00<?, ?it/s]

Saved: /kaggle/working/dpfn_graph_pkls_final/dev_aff_adj.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/dev_con_graph_adj_k5.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/dev_top5_concepts_k5.pkl


Build PKLs: test:   0%|          | 0/1037 [00:00<?, ?it/s]

Saved: /kaggle/working/dpfn_graph_pkls_final/test_aff_adj.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/test_con_graph_adj_k5.pkl
Saved: /kaggle/working/dpfn_graph_pkls_final/test_top5_concepts_k5.pkl


In [11]:
def inspect(split, idx=0, k=5):
    with open(OUT/f"{split}_aff_adj.pkl","rb") as f:
        aff = pickle.load(f)
    with open(OUT/f"{split}_con_graph_adj_k{k}.pkl","rb") as f:
        con = pickle.load(f)
    with open(OUT/f"{split}_top5_concepts_k{k}.pkl","rb") as f:
        top = pickle.load(f)

    A_aff = np.array(aff[idx])
    A_con = np.array(con[idx])

    print("split:", split, "idx:", idx)
    print("A_aff shape:", A_aff.shape, "min/max:", float(A_aff.min()), float(A_aff.max()))
    print("A_con shape:", A_con.shape, "min/max:", float(A_con.min()), float(A_con.max()))
    print("top5 concepts:", top[idx])

inspect("train", 0, k=5)


split: train idx: 0
A_aff shape: (10, 10) min/max: 0.3545415997505188 1.6349223852157593
A_con shape: (15, 15) min/max: 0.0 1.0
top5 concepts: ['chucker', 'chucks', 'chuckie', 'chucky', 'guffaw']
